# Cleaning the FRED data

**Goal:** turn 6 separate FRED files into ONE table with one row per month, ready for Power BI.

**The problem:** the files update at different speeds.

| File | What it is | How often |
|---|---|---|
| MORTGAGE30US | 30-year mortgage rate | weekly |
| DGS10 | 10-year Treasury yield | daily |
| FEDFUNDS | Fed funds rate | monthly |
| UNRATE | Unemployment rate | monthly |
| CPIAUCSL | Consumer price index | monthly |
| DRSFRMACBS | Mortgage delinquency rate | quarterly |

**The fix:** put everything on a monthly scale, then join the files together.

**Result:** `data/fred_clean.csv`

## Step 1: Load the six files

In [1]:
import pandas as pd

# Each file has two columns: the date and the value.
# index_col sets the date as the row label, which makes joining easy later.
mortgage = pd.read_csv("data/MORTGAGE30US.csv", index_col="observation_date", parse_dates=True)
treasury = pd.read_csv("data/DGS10.csv",         index_col="observation_date", parse_dates=True)
fedfunds = pd.read_csv("data/FEDFUNDS.csv",      index_col="observation_date", parse_dates=True)
unrate   = pd.read_csv("data/UNRATE.csv",        index_col="observation_date", parse_dates=True)
cpi      = pd.read_csv("data/CPIAUCSL.csv",      index_col="observation_date", parse_dates=True)
delinq   = pd.read_csv("data/DRSFRMACBS.csv",    index_col="observation_date", parse_dates=True)

# Quick look at the weekly mortgage file
mortgage.head()

,MORTGAGE30US
observation_date,
1971-04-02,7.33
1971-04-09,7.31
1971-04-16,7.31
1971-04-23,7.31
1971-04-30,7.29


## Step 2: Fix missing values

FRED uses a "." when a value is missing (for example, bond market holidays). Pandas reads that as text, so we convert it to a number. A "." becomes an empty value (NaN).

In [2]:
# Convert each file's value column from text to numbers
treasury["DGS10"] = pd.to_numeric(treasury["DGS10"], errors="coerce")
unrate["UNRATE"]  = pd.to_numeric(unrate["UNRATE"],  errors="coerce")
cpi["CPIAUCSL"]   = pd.to_numeric(cpi["CPIAUCSL"],   errors="coerce")

# How many missing values are there?
print("Treasury missing days:", treasury["DGS10"].isna().sum())
print("Unemployment missing months:", unrate["UNRATE"].isna().sum())
print("CPI missing months:", cpi["CPIAUCSL"].isna().sum())

Treasury missing days: 720
Unemployment missing months: 1
CPI missing months: 1


## Step 3: Convert everything to monthly

- **Daily and weekly** data: average all the values in each month.
- **Monthly** data: already monthly, so we just line up the dates.
- **Quarterly** delinquency: repeat each quarter's value for its 3 months.

`"MS"` means "month start", so every month is dated the 1st (for example 2022-05-01).

In [3]:
# Daily / weekly -> monthly average
mortgage_m = mortgage["MORTGAGE30US"].resample("MS").mean()
treasury_m = treasury["DGS10"].resample("MS").mean()

# Already monthly (this just makes the dates line up)
fedfunds_m = fedfunds["FEDFUNDS"].resample("MS").mean()
unrate_m   = unrate["UNRATE"].resample("MS").mean()
cpi_m      = cpi["CPIAUCSL"].resample("MS").mean()

# Quarterly -> repeat the value for the next 2 months as well
delinq_m = delinq["DRSFRMACBS"].resample("MS").ffill(limit=2)

mortgage_m.tail()

observation_date
2026-05-01    6.4425
2026-06-01    6.4900
2026-07-01    6.5420
2026-08-01    6.6675
2026-09-01    6.8625
Freq: MS, Name: MORTGAGE30US, dtype: float64

## Step 4: Combine into one table

In [4]:
df = pd.DataFrame({
    "MORTGAGE30US": mortgage_m,
    "FEDFUNDS":     fedfunds_m,
    "DGS10":        treasury_m,
    "UNRATE":       unrate_m,
    "CPIAUCSL":     cpi_m,
    "DRSFRMACBS":   delinq_m,
})

# The delinquency data starts in 1991, so start everything there
df = df.loc["1991-01-01":]
df.head()

,MORTGAGE30US,FEDFUNDS,DGS10,UNRATE,CPIAUCSL,DRSFRMACBS
observation_date,,,,,,
1991-01-01,9.6375,6.91,8.091905,6.4,134.7,3.10
1991-02-01,9.3650,6.25,7.854737,6.6,134.8,3.10
1991-03-01,9.5000,6.12,8.110000,6.8,134.8,3.10
1991-04-01,9.4925,5.91,8.039091,6.7,135.1,3.17
1991-05-01,9.4720,5.78,8.067727,6.9,135.6,3.17


## Step 5: Add two calculated columns

1. **Inflation** = how much CPI grew compared to the same month one year ago, in percent.
2. **Mortgage spread** = mortgage rate minus the 10-year Treasury yield. Banks watch this as a measure of lending risk.

In [5]:
# pct_change(12) compares each month to the month 12 rows earlier.
# We calculate it on the full CPI history so that 1991 also gets a value.
inflation = cpi_m.pct_change(12) * 100
df["Inflation_YoY"] = inflation

df["Mortgage_Spread"] = df["MORTGAGE30US"] - df["DGS10"]

df[["MORTGAGE30US", "DGS10", "Mortgage_Spread", "Inflation_YoY"]].tail()

,MORTGAGE30US,DGS10,Mortgage_Spread,Inflation_YoY
observation_date,,,,
2026-05-01,6.4425,4.484000,1.958500,4.166615
2026-06-01,6.4900,4.470476,2.019524,3.463531
2026-07-01,6.5420,4.599545,1.942455,3.303856
2026-08-01,6.6675,4.684286,1.983214,3.353016
2026-09-01,6.8625,4.958947,1.903553,NaN


## Step 6: Deal with gaps

- **October 2025** has no unemployment or CPI (the government did not publish them), so we remove that month instead of guessing.
- The **last few months of delinquency** are blank because that number is released late. We leave them blank.

In [6]:
rows_before = len(df)

# Drop any month missing one of the main columns
df = df.dropna(subset=["MORTGAGE30US", "FEDFUNDS", "DGS10", "UNRATE", "CPIAUCSL", "Inflation_YoY"])

print("Rows before:", rows_before)
print("Rows after: ", len(df))
print()
print("Blank values left in each column:")
print(df.isna().sum())

Rows before: 429
Rows after:  427

Blank values left in each column:
MORTGAGE30US       0
FEDFUNDS           0
DGS10              0
UNRATE             0
CPIAUCSL           0
DRSFRMACBS         4
Inflation_YoY      0
Mortgage_Spread    0
dtype: int64


## Step 7: Save the result

In [7]:
df = df.round(3)
df.index.name = "Date"
df.to_csv("data/fred_clean.csv")

print("Saved", len(df), "rows from", df.index.min().strftime("%Y-%m"), "to", df.index.max().strftime("%Y-%m"))
df.tail()

Saved 427 rows from 1991-01 to 2026-08


,MORTGAGE30US,FEDFUNDS,DGS10,UNRATE,CPIAUCSL,DRSFRMACBS,Inflation_YoY,Mortgage_Spread
Date,,,,,,,,
2026-04-01,6.332,3.64,4.319,4.3,332.407,1.86,3.779,2.013
2026-05-01,6.442,3.63,4.484,4.3,333.979,NaN,4.167,1.958
2026-06-01,6.490,3.63,4.470,4.2,332.568,NaN,3.464,2.020
2026-07-01,6.542,3.63,4.600,4.1,332.813,NaN,3.304,1.942
2026-08-01,6.668,3.63,4.684,4.1,334.131,NaN,3.353,1.983
